### Purpose
Core revenue and product review analysis on Olist transactional marts.

Read data from `marts.fact_orders`, `marts.dim_product`, and `marts.fact_order_item`.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

BASE_DIR = Path.cwd().parent

# Read environment variables
db_path_env = os.getenv("DB_PATH")

if db_path_env is None:
    raise ValueError("DB_PATH not found in .env")

DB_PATH = BASE_DIR / db_path_env

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("muted")

con = duckdb.connect(str(DB_PATH), read_only=True)

### Chart - Monthly revenue over time

In [ ]:
df = con.execute("""
    SELECT
        DATE_TRUNC('month', order_date) AS month,
        SUM(total_order_value) AS total_revenue
    FROM marts.fact_orders
    WHERE order_status = 'delivered'
    GROUP BY 1
    ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    df['month'],
    df['total_revenue'],
    marker='o',
    linewidth=2,
    color='steelblue'
)

ax.set_title(
    'Monthly Revenue - Delivered Orders',
    fontsize=14,
    pad=12
)

ax.set_xlabel('Month')
ax.set_ylabel('Revenue (BRL)')

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f'R${x:,.0f}')
)

plt.xticks(rotation=45)
plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "monthly_revenue.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()


 `Revenue peaks sharply in November 2017, consistent with Brazilian Black Friday. The business should pre-build inventory by October and negotiate seller capacity in Q3 to avoid stockouts during the spike.`

### Chart - Top 15 products by revenue

In [ ]:
df = con.execute("""
    SELECT
        category_english AS category,
        SUM(price) AS total_revenue
    FROM marts.fact_order_item
    WHERE category_english IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 15
""").df()

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(df['category'], df['total_revenue'])
ax.bar_label(bars, fmt='R$%.0f', padding=4, fontsize=10)

ax.set_title('Top 15 Product Categories by Revenue', fontsize=14, pad=12)
ax.set_xlabel('Total Revenue (BRL)')
ax.invert_yaxis()

plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "top_categories.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()

`The top 3 categories account for ~35% of all revenue - a classic 80/20 concentration. Marketing and inventory investment should be weighted toward these categories first.`

### Chart - Average review score by product category

In [ ]:
df = con.execute("""
    SELECT
        foi.category_english AS category,
        COUNT(DISTINCT fo.order_id) AS total_orders,
        ROUND(AVG(fo.review_score), 2) AS avg_review_score
    FROM marts.fact_orders fo
    INNER JOIN marts.fact_order_item foi
        ON fo.order_id = foi.order_id
    WHERE fo.order_status = 'delivered'
      AND fo.review_score IS NOT NULL
      AND foi.category_english IS NOT NULL
    GROUP BY 1
    HAVING COUNT(DISTINCT fo.order_id) > 100
    ORDER BY avg_review_score DESC
    LIMIT 20
""").df()

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    df['category'],
    df['avg_review_score'],
    color='mediumpurple'
)

ax.bar_label(
    bars,
    fmt='%.2f',
    padding=4,
    fontsize=10
)

ax.set_title(
    'Average Review Score by Product Category',
    fontsize=14,
    pad=12
)

ax.set_xlabel('Average Review Score')
ax.set_xlim(3.5, 4.8)
ax.set_ylabel('Product Category')

ax.invert_yaxis()

plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "avg_review_by_category.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()

`Lower-rated categories may require operational improvements related to delivery quality, product expectations, or seller performance.`

### Chart - Order value distribution

In [ ]:
df = con.execute("""
    SELECT total_order_value
    FROM marts.fact_orders
    WHERE order_status = 'delivered'
      AND total_order_value < 1000
""").df()

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    df['total_order_value'],
    bins=50,
    color='steelblue',
    edgecolor='white'
)

ax.axvline(
    df['total_order_value'].median(),
    color='red',
    linestyle='--',
    label=f"Median: R${df['total_order_value'].median():.0f}"
)

ax.set_title(
    'Order Value Distribution (Capped at R$1000)',
    fontsize=14,
    pad=12
)

ax.set_xlabel('Order Value (BRL)')
ax.set_ylabel('Number of Orders')

ax.legend()

plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "order_distribution.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()

`Order values are right-skewed - the majority of orders are below R$200. Upsell campaigns targeting the R$100–200 segment could meaningfully raise average order value without needing new customers.`

### Chart - Revenue by state

In [ ]:
df = con.execute("""
    SELECT
        customer_state AS state,
        SUM(total_order_value) AS total_revenue
    FROM marts.fact_orders
    WHERE order_status = 'delivered'
      AND customer_state IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 15
""").df()

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    df['state'],
    df['total_revenue'],
    color='steelblue'
)

ax.bar_label(
    bars,
    fmt='R$%.0f',
    padding=4,
    fontsize=9,
    rotation=45
)

ax.set_title(
    'Revenue by Customer State - Top 10',
    fontsize=14,
    pad=12
)

ax.set_xlabel('State')
ax.set_ylabel('Total Revenue (BRL)')

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M')
)

plt.tight_layout()

output_dir = BASE_DIR / "dashboard" / "screenshots"
output_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    output_dir / "revenue_by_state.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()

`SP alone generates over 40% of revenue. Logistics partnerships in MG and RJ - the next two states - could unlock material growth at lower customer acquisition cost than expanding to lower-volume states.`